In [ ]:
# -*- coding: utf-8 -*-

from pathlib import Path
from datetime import datetime
import time
import pandas as pd

# ==========================
# 設定
# ==========================
OUTPUT_DIR = Path(r"D:\musashino-university\finance\change_point_output")

TARGET_RUN_ID = "20260510_142302"

METHODS = ["pelt", "binseg", "bottomup", "dynp", "window"]
MODELS  = ["l1", "l2", "rbf", "normal", "linear"]

DATETIME_COL = "cp_datetime"

SAVE_ENCODING = "utf-8-sig"

OUT_PER_COMBO = OUTPUT_DIR / f"change_point_daily_counts_25combos_{TARGET_RUN_ID}.csv"
OUT_PIVOT     = OUTPUT_DIR / f"change_point_daily_counts_pivot_25combos_{TARGET_RUN_ID}.csv"
OUT_TOTAL     = OUTPUT_DIR / f"change_point_daily_counts_total_25combos_{TARGET_RUN_ID}.csv"


# ==========================
# ログ関数
# ==========================
START_TIME = time.time()

def log(msg):
    elapsed = time.time() - START_TIME
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now} | +{elapsed:,.1f}s] {msg}")


# ==========================
# メイン処理
# ==========================
all_daily = []

log("日別変化点カウントを開始します。")

combo_no = 0
total_combo = len(METHODS) * len(MODELS)

for method in METHODS:
    for model in MODELS:
        combo_no += 1

        file_path = OUTPUT_DIR / f"change_point_pairs_{method}_{model}_{TARGET_RUN_ID}.csv"

        log("=" * 70)
        log(f"[{combo_no}/{total_combo}] method={method}, model={model}")
        log(f"読み込み対象: {file_path.name}")

        if not file_path.exists():
            log("ファイルが存在しないためスキップします。")
            continue

        try:
            df = pd.read_csv(file_path, encoding="utf-8-sig")
        except pd.errors.EmptyDataError:
            log("空ファイルのためスキップします。")
            continue
        except Exception as e:
            log(f"読み込み失敗: {e}")
            continue

        if df.empty:
            log("データ行が0件のためスキップします。")
            continue

        if DATETIME_COL not in df.columns:
            log(f"{DATETIME_COL} 列がありません。列一覧: {list(df.columns)}")
            continue

        df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce")
        df = df.dropna(subset=[DATETIME_COL])

        if df.empty:
            log("日時変換後の有効データが0件のためスキップします。")
            continue

        df["date"] = df[DATETIME_COL].dt.date

        daily = (
            df.groupby("date")
              .size()
              .reset_index(name="change_point_count")
        )

        daily["method"] = method
        daily["model"] = model
        daily["combo"] = f"{method}_{model}"
        daily["run_id"] = TARGET_RUN_ID

        daily = daily[
            ["run_id", "method", "model", "combo", "date", "change_point_count"]
        ]

        all_daily.append(daily)

        log(f"読み込み行数: {len(df):,}")
        log(f"日別集計行数: {len(daily):,}")
        log(f"変化点合計: {daily['change_point_count'].sum():,}")


# ==========================
# 保存
# ==========================
if not all_daily:
    raise ValueError("集計できるCSVがありませんでした。")

result = pd.concat(all_daily, ignore_index=True)

result["date"] = pd.to_datetime(result["date"])
result = result.sort_values(["method", "model", "date"])

result.to_csv(OUT_PER_COMBO, index=False, encoding=SAVE_ENCODING)

log("=" * 70)
log(f"25通り別の日別カウントCSVを保存しました: {OUT_PER_COMBO}")

# 日付 × 組み合わせ の横持ち表
pivot = (
    result.pivot_table(
        index="date",
        columns="combo",
        values="change_point_count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

pivot.to_csv(OUT_PIVOT, index=False, encoding=SAVE_ENCODING)
log(f"ピボット表CSVを保存しました: {OUT_PIVOT}")

# 25通り合計の日別カウント
total_daily = (
    result.groupby("date")["change_point_count"]
          .sum()
          .reset_index(name="total_change_point_count")
          .sort_values("date")
)

total_daily.to_csv(OUT_TOTAL, index=False, encoding=SAVE_ENCODING)
log(f"25通り合計の日別カウントCSVを保存しました: {OUT_TOTAL}")

log("=" * 70)
log("処理完了")
log(f"対象組み合わせ数: {result['combo'].nunique()} / 25")
log(f"全変化点数: {result['change_point_count'].sum():,}")
log(f"総処理時間: {time.time() - START_TIME:,.1f}s")

print()
print("=== 出力ファイル ===")
print(OUT_PER_COMBO)
print(OUT_PIVOT)
print(OUT_TOTAL)